In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/crop_loss_master_all.csv", low_memory=False)
print(df.shape)
df.head()

(75077, 94)


,household_id,crop_code,crop_name,crop_domain,disposition_share1_pct_unconfirmed,disposition_share2_pct_unconfirmed,disposition_share3_pct_unconfirmed,disposition_share4_pct_unconfirmed,disposition_share5_pct_unconfirmed,disposition_share6_pct_unconfirmed,...,woreda_code,is_rural,survey_year,region_name,rainfall_belg_mm,rainfall_belg_pct_of_avg,rainfall_meher_mm,rainfall_meher_pct_of_avg,rainfall_annual_mm,rainfall_annual_pct_of_avg
0,1.010102e+12,2.0,MAIZE,field_crop,95.0,5.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2011,Tigray,120.648096,112.0,643.046036,95.0,800.284077,96.3
1,1.010102e+12,3.0,MILLET,field_crop,95.0,5.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2011,Tigray,120.648096,112.0,643.046036,95.0,800.284077,96.3
2,1.010102e+12,6.0,SORGHUM,field_crop,95.0,5.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2011,Tigray,120.648096,112.0,643.046036,95.0,800.284077,96.3
3,1.010102e+12,38.0,RED PEPPER,garden_crop,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,2011,Tigray,120.648096,112.0,643.046036,95.0,800.284077,96.3
4,1.010102e+12,2.0,MAIZE,field_crop,95.0,5.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2011,Tigray,120.648096,112.0,643.046036,95.0,800.284077,96.3


In [3]:
pd.set_option("display.max_rows", 100)
missing = df.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(1)
missing.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
loss_detail_ka_unconfirmed,75001,99.9
loss_detail_la_unconfirmed,75001,99.9
loss_detail_z_unconfirmed,75002,99.9
loss_detail_x_unconfirmed,74982,99.9
loss_detail_w_unconfirmed,74975,99.9
loss_detail_v_unconfirmed,74968,99.9
loss_detail_pa_unconfirmed,75007,99.9
loss_detail_oa_unconfirmed,75011,99.9
loss_detail_na_unconfirmed,75008,99.9
loss_detail_t_unconfirmed,74904,99.8


## Defining the target, then dropping leakage / unreliable columns

`total_loss_qty` (and the `loss_reason1/2/3_occurred/unit/qty` columns it's
built from) can only be used to build the **target** — using them as
**features** would be leakage, since they directly encode the outcome being
predicted. Built `loss_occurred` from `total_loss_qty` first, then those
columns (plus everything else too sparse/ambiguous to trust — see below) are
dropped from the feature set entirely.

In [4]:
# Target: was any loss recorded for this household+crop+wave row.
df["loss_occurred"] = (df["total_loss_qty"].fillna(0) > 0).astype(int)
print(df["loss_occurred"].value_counts(normalize=True).rename("share").round(3))

# Leakage: total_loss_qty and everything it's built from.
LEAKAGE_COLS = [
    "total_loss_qty",
    "loss_reason1_occurred", "loss_reason1_unit", "loss_reason1_qty",
    "loss_reason2_occurred", "loss_reason2_unit", "loss_reason2_qty",
    "loss_reason3_occurred", "loss_reason3_unit", "loss_reason3_qty",
]

# loss_detail_* / loss_extra_*: 70-99.9% missing, and several describe the
# circumstances of the loss itself (leakage risk on top of sparsity).
LOSS_DETAIL_COLS = [c for c in df.columns if c.startswith("loss_detail_") or c.startswith("loss_extra_")]

# storage_*_unconfirmed: 69-98% missing, too sparse to trust.
STORAGE_UNCONFIRMED_COLS = [c for c in df.columns if c.startswith("storage_")]

# disposition_*: 45-46% missing (absent entirely for Waves 2018/2021) and
# ambiguous timing relative to the loss event.
DISPOSITION_COLS = [c for c in df.columns if c.startswith("disposition_")]

# Redundant with crop_code/crop_name identity, and 35% missing.
CROP_DOMAIN_COLS = ["crop_domain"]

# zone_code is a per-region local sequence, not nationally unique/comparable
# without pairing to region_code -- not usable as-is.
GEO_COLS = ["zone_code", "woreda_code"]

# Semantics unconfirmed, 82-84% missing.
UNCONFIRMED_MISC_COLS = ["ph_saq07", "ph_saq07_loss"]

DROP_COLS = (
    LEAKAGE_COLS + LOSS_DETAIL_COLS + STORAGE_UNCONFIRMED_COLS
    + DISPOSITION_COLS + CROP_DOMAIN_COLS + GEO_COLS + UNCONFIRMED_MISC_COLS
)
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

print(f"\ndropping {len(DROP_COLS)} columns")
df_model = df.drop(columns=DROP_COLS)
print("remaining shape:", df_model.shape)
print(list(df_model.columns))

loss_occurred
0    0.935
1    0.065
Name: share, dtype: float64

dropping 80 columns
remaining shape: (75077, 15)
['household_id', 'crop_code', 'crop_name', 'household_size', 'region_code', 'is_rural', 'survey_year', 'region_name', 'rainfall_belg_mm', 'rainfall_belg_pct_of_avg', 'rainfall_meher_mm', 'rainfall_meher_pct_of_avg', 'rainfall_annual_mm', 'rainfall_annual_pct_of_avg', 'loss_occurred']


## Check for missing values

In [5]:
missing_val = df_model.isna().sum().to_frame("missing_count").sort_values("missing_count", ascending=False)
missing_val


,missing_count
crop_name,3883
crop_code,1024
household_size,381
region_code,381
is_rural,381
region_name,381
rainfall_belg_mm,381
rainfall_belg_pct_of_avg,381
rainfall_meher_mm,381
rainfall_meher_pct_of_avg,381


## Missing values in percentage

In [7]:
missing_val_percent = (missing_val / len(df_model) * 100).round(1)
missing_val_percent

,missing_count
crop_name,5.2
crop_code,1.4
household_size,0.5
region_code,0.5
is_rural,0.5
region_name,0.5
rainfall_belg_mm,0.5
rainfall_belg_pct_of_avg,0.5
rainfall_meher_mm,0.5
rainfall_meher_pct_of_avg,0.5
